In [1]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# 05_rq3_sample_creation.py
# Purpose of Script: Generate Samples for Regression Analysis
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initialization ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Google Drive
#~~~~~~~~~~~~~~~~~~~~~~~~~~
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Libraries
#~~~~~~~~~~~~~~~~~~~~~~~~~~
import numpy as np
import pandas as pd
import gc
import duckdb
from pathlib import Path

In [3]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Print Versions
#~~~~~~~~~~~~~~~~~~~~~~~~~~
print(f"Numpy version = {np.__version__}")
print(f"Pandas version = {pd.__version__}")
print(f"DuckDB version = {duckdb.__version__}")

Numpy version = 2.0.2
Pandas version = 2.2.2
DuckDB version = 1.3.2


In [4]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initiate Duck Connection
#~~~~~~~~~~~~~~~~~~~~~~~~~~
con = duckdb.connect()

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Input/Output Paths
#~~~~~~~~~~~~~~~~~~~~~~~~~~
### Input
path_samp = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/03_samples/"
path_clean = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/02_clean/"
path_out = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/03_outputs/"

#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Seed
#~~~~~~~~~~~~~~~~~~~~~~~~~~
random_seed = 42

In [5]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Generate Regression Samples ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Base Directory
dir_base = Path(path_clean)

# Define Sample %
samp_perc = 0.1

# List All Files
files_facebook = sorted(dir_base.glob("*-facebook/facebook.parquet"))
files_instagram = sorted(dir_base.glob("*-instagram/instagram.parquet"))

In [ ]:
# Sample All Files
samples = []

# All Files
files = {"facebook": files_facebook, "instagram" : files_instagram}

# Start File
file_num = 1

# Loop
for platform, platform_files in files.items():

    # Print Progress
    print(f"Processing Platform: {platform}")

    # Drop Temporary Table
    con.execute("drop table if exists sample")

    # Column Selection
    if platform in ["facebook", "instagram"]:
        cols_to_select = """ p_name, date, cont_type, source, cat,
                             cat_spec, cat_spec_other, des_ground,
                             illegal_c_ground , illegal_c_ex"""
    else:
        cols_to_select = """ p_name, date, cont_type, source, cat,
            cat_spec, cat_spec_other, des_ground,
            des_fact, illegal_c_ground, illegal_c_ex,
            incomp_c_ground, incomp_c_ex, incomp_c_illegal,
            des_vis, des_vis_other, des_vis_end_date,
            des_mon, des_mon_other, des_mon_end_date,
            des_prov, des_prov_end_date,
            des_acc, des_acc_end_date """

    first = True

    for file in platform_files:

        # Print Progress
        print(f"Processing File: {file}")

        if first:

            con.execute(f""" create table sample as
                             select {cols_to_select}
                             from read_parquet('{file}')
                             using sample {samp_perc} percent (system, {random_seed})""")

            first = False

        else:

            con.execute(f""" insert into sample
                             select {cols_to_select}
                             from read_parquet('{file}')
                             using sample {samp_perc} percent (system, {random_seed})""")

    con.execute(f""" copy sample
                     to '{path_out}05_rq3_{file_num}_sample_prep_{platform}.parquet'
                     (format parquet)""")

    # Increase File Number
    file_num += 1

    # Clean Results
    con.execute(f""" drop table sample""")
    con.commit()
    con.close()
    con = duckdb.connect("/content/working.duckdb")

In [9]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Identify Synthetic Media in Platforms ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Synthetic Media Identified only by quantiative column
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Facebook - Quant Only
#~~~~~~~~~~~~~~~~~~~~~~~~~~
df_facebook = con.execute(f""" select * from '{path_out}05_rq3_1_sample_prep_facebook.parquet'""").df()

# Format Illegal Columns
df_facebook["illegal_flag_ground"] = np.where(df_facebook["illegal_c_ground"].isna() , 0, 1)
df_facebook["illegal_flag_ex"] = np.where(df_facebook["illegal_c_ex"].isna() , 0, 1)
df_facebook["illegal_flag"] = np.where((df_facebook["illegal_flag_ground"] == 1) | (df_facebook["illegal_flag_ex"] == 1), 1, 0)

# Set Synth Flag
df_facebook["synth_flag"] = np.where(df_facebook["cont_type"] == "sm", 1, 0)

# Introduce Date Features
sample_start = pd.Timestamp("2025-01-01")
df_facebook["day_since_start"] = (df_facebook["date"] - sample_start).dt.days
df_facebook["doy"] = df_facebook["date"].dt.dayofyear
df_facebook["month"] = df_facebook["date"].dt.month

# Cut-Down Table
df_facebook = df_facebook[["synth_flag","p_name","illegal_flag","doy","month",
                           "day_since_start","source","cat","cat_spec"]]

# Remove Historic or Empty Category
df_facebook = df_facebook[df_facebook["cat"] != "historic_or_empty"]

In [14]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Instagram - Quant Only
#~~~~~~~~~~~~~~~~~~~~~~~~~~
df_inst = con.execute(f""" select * from '{path_out}05_rq3_2_sample_prep_instagram.parquet'""").df()

# Format Illegal Columns
df_inst["illegal_flag_ground"] = np.where(df_inst["illegal_c_ground"].isna() , 0, 1)
df_inst["illegal_flag_ex"] = np.where(df_inst["illegal_c_ex"].isna() , 0, 1)
df_inst["illegal_flag"] = np.where((df_inst["illegal_flag_ground"] == 1) | (df_inst["illegal_flag_ex"] == 1), 1, 0)

# Set Synth Flag
df_inst["synth_flag"] = np.where(df_inst["cont_type"] == "sm", 1, 0)

# Introduce Date Features
sample_start = pd.Timestamp("2025-01-01")
df_inst["day_since_start"] = (df_inst["date"] - sample_start).dt.days
df_inst["doy"] = df_inst["date"].dt.dayofyear
df_inst["month"] = df_inst["date"].dt.month

# Cut-Down Table
df_inst = df_inst[["synth_flag","p_name","illegal_flag","doy","month",
                           "day_since_start","source","cat","cat_spec"]]

# Remove Historic or Empty
df_inst = df_inst[df_inst["cat"] != "historic_or_empty"]

In [16]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Generate Full Regression Sample ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Concatenate All Platforms
df_reg = pd.concat([df_facebook, df_inst], ignore_index=True).reset_index(drop=True)

# Export
df_reg.to_parquet(f"{path_out}05_rq3_5_regression_sample.parquet")